### Natural Language Processing

*Natural language processing* (NLP) combines linguistics and machine learning to perform tasks such as sentiment analysis, language translation, named entity recognition, and text summarization

#### Example 1

We'll use the newsgroup dataset to illustrate tokenization with `CountVectorizer()` and `TfidfVectorizer()`

In [1]:
# allow multiple outputs in one cell
if True:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = "all"

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [19]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

In [3]:
categories = ['alt.atheism', 'soc.religion.christian']
data = fetch_20newsgroups(subset='train', categories=categories)

print(f"Total samples: {len(data.data)}")
print(f"Class names: {data.target_names}")
print(f"Class distribution: {np.unique(data.target, return_counts=True)}")

print(f"\nFirst example in category {data.target_names[data.target[0]]}:\n")
print(f"{data.data[0]}")

Total samples: 1079
Class names: ['alt.atheism', 'soc.religion.christian']
Class distribution: (array([0, 1]), array([480, 599]))

First example in category soc.religion.christian:

From: nigel.allen@canrem.com (Nigel Allen)
Subject: library of congress to host dead sea scroll symposium april 21-22
Lines: 96


 Library of Congress to Host Dead Sea Scroll Symposium April 21-22
 To: National and Assignment desks, Daybook Editor
 Contact: John Sullivan, 202-707-9216, or Lucy Suddreth, 202-707-9191
          both of the Library of Congress

   WASHINGTON, April 19  -- A symposium on the Dead Sea 
Scrolls will be held at the Library of Congress on Wednesday,
April 21, and Thursday, April 22.  The two-day program, cosponsored
by the library and Baltimore Hebrew University, with additional
support from the Project Judaica Foundation, will be held in the
library's Mumford Room, sixth floor, Madison Building.
   Seating is limited, and admission to any session of the symposium
must be requested

* `CountVectorizer(binary=True)` creates a vectorizer that marks word presence as 1 regardless of frequency (each word is simply present or absent)
* `vec.fit()` learns the unique vocabulary by scanning all text entries
* `vec.transform()` converts text to a numerical matrix where each row represents a text entry and each column represents a word's presence/absence
* `X[1].toarray()` displays the binary vector for a specific text entry, showing which vocabulary words are present

In [4]:
vec = CountVectorizer(binary=True)
vec.fit(data.data)
X = vec.transform(data.data)
y = data.target

pd.DataFrame(data=X.toarray(), columns=vec.get_feature_names_out())
vec.get_feature_names_out()[100:120]  # extract some range of words

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


,00,000,0001,000406,001125,0014,00630,01,0100,010116,...,zoo,zorasterism,zues,zumder,zur,zurlo,zus,zvonko,zwart,zyklon
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1075,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1076,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1077,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


array(['10yo', '11', '110', '111', '111651', '111713', '112', '112127',
       '1122', '112329', '112430', '11292', '113', '113255', '11361',
       '114', '114127', '114133', '114140', '115'], dtype=object)

Since some of the words appear only rarely, we can only keep the subset that appears at least `min_df` times and ignore common English words ("the", "and", etc.)

In [5]:
vec = CountVectorizer(binary=True, min_df=50, stop_words='english')
vec.fit(data.data)
X = vec.transform(data.data)
y = data.target

pd.DataFrame(data=X.toarray(), columns=vec.get_feature_names_out())
vec.get_feature_names_out()[100:120]  # extract some range of words

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


,00,01,03,10,11,12,13,14,15,16,...,works,world,wouldn,wpd,writes,written,wrong,wrote,years,yes
0,0,0,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,0,0,0,0
2,0,1,0,0,1,0,1,0,0,0,...,0,0,0,0,1,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,1,0,0,1
4,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1075,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1076,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1077,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


array(['comes', 'coming', 'common', 'computer', 'conclusion', 'consider',
       'considered', 'correct', 'couple', 'course', 'created', 'cs',
       'cwru', 'david', 'day', 'days', 'dead', 'death', 'definition',
       'deleted'], dtype=object)

Here are some other options which are useful for `CountVectorizer()`

In [16]:
print("Words appearing in at least 5 but fewer than 100 examples:",
      CountVectorizer(min_df=5, max_df=100).fit_transform(data.data).shape, "(examples, words)")

print("Same as above, but limited to 1000 most frequent words:",
      CountVectorizer(min_df=5, max_df=100, max_features=1000).fit_transform(data.data).shape, "(examples, words)")

vocab = ["good", "bad", "silly", "horrible"]
print(f"Words when using a user-defined vocabulary {vocab}:",
      CountVectorizer(vocabulary=vocab).fit_transform(data.data).shape, "(examples, words)")

print("Words when considering both unigrams and bigraphs with min_df=5, max_df=100, max_features=1000, common English words removed:")
vec = CountVectorizer(ngram_range=(1,2), min_df=5, max_df=100, max_features=1000, stop_words="english")
X = vec.fit_transform(data.data)
pd.DataFrame(data=X.toarray(), columns=vec.get_feature_names_out())

Words appearing in at least 5 but fewer than 100 examples: (1079, 4803) (examples, words)
Same as above, but limited to 1000 most frequent words: (1079, 1000) (examples, words)
Words when using a user-defined vocabulary ['good', 'bad', 'silly', 'horrible']: (1079, 4) (examples, words)
Words when considering both unigrams and bigraphs with min_df=5, max_df=100, max_features=1000, common English words removed:


,00,000,01,02,03,05,11,12,13,16,...,wpd sgi,write,writing,written,wwc,wwc edu,year,young,yoyo,yoyo cc
0,0,0,0,0,0,0,1,0,1,0,...,0,0,2,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,1,1,1,0,0,0,0
2,0,0,1,0,0,0,1,0,1,0,...,0,0,0,1,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,1,0,0,1,0,0,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1075,0,4,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1076,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1077,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


We can use `TfidfVectorizer()` to convert a collection of text documents into a matrix of TFIDF (Term Frequency-Inverse Document Frequency) features which reflect how important a word is in a document relative to the entire corpus. Effectively, if a word appears in many documents, it is considered less important

In [28]:
# create an artificial dataset of text examples
corpus = [
   "This is the first document, the FIRST",
   "This is the second document",
   "This is the third document",
   "This is the fourth document"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
header = vectorizer.get_feature_names_out()
labels = ['D1', 'D2', 'D3', 'D4']
print("With CountVectorizer (simply counts how many times each word appears):")
pd.DataFrame(X.toarray(), columns = header, index = labels)

# measure the relative importance of each word
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)
header = vectorizer.get_feature_names_out()
labels = ['D1', 'D2', 'D3', 'D4']
print("With TfidfVectorizer (measures the relative importance of each word):")
pd.DataFrame(X.toarray(), columns = header, index = labels)

With CountVectorizer (simply counts how many times each word appears):


,document,first,fourth,is,second,the,third,this
D1,1,2,0,1,0,2,0,1
D2,1,0,0,1,1,1,0,1
D3,1,0,0,1,0,1,1,1
D4,1,0,1,1,0,1,0,1


With TfidfVectorizer (measures the relative importance of each word):


,document,first,fourth,is,second,the,third,this
D1,0.214725,0.822953,0.000000,0.214725,0.000000,0.429451,0.000000,0.214725
D2,0.361028,0.000000,0.000000,0.361028,0.691835,0.361028,0.000000,0.361028
D3,0.361028,0.000000,0.000000,0.361028,0.000000,0.361028,0.691835,0.361028
D4,0.361028,0.000000,0.691835,0.361028,0.000000,0.361028,0.000000,0.361028


#### Example 2

We'll use the newsgroup dataset to illustrate text analysis with `CountVectorizer()` and `LogisticRegression()` via a pipeline with hyperparameter optimization

In [ ]:
# allow multiple outputs in one cell
if True:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = "all"

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [53]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

In [54]:
categories = ['alt.atheism', 'soc.religion.christian']
data = fetch_20newsgroups(subset='train', categories=categories)

print(f"Total samples: {len(data.data)}")
print(f"Class names: {data.target_names}")
print(f"Class distribution: {np.unique(data.target, return_counts=True)}")

X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.3, random_state=42)
print("Class breakdown [train] [test]: ", np.unique(y_train, return_counts=True)[1], np.unique(y_test, return_counts=True)[1])

print(f"\nFirst example in category {data.target_names[data.target[0]]}:\n")
print(f"{data.data[0]}")

Total samples: 1079
Class names: ['alt.atheism', 'soc.religion.christian']
Class distribution: (array([0, 1]), array([480, 599]))
Class breakdown [train] [test]:  [334 421] [146 178]

First example in category soc.religion.christian:

From: nigel.allen@canrem.com (Nigel Allen)
Subject: library of congress to host dead sea scroll symposium april 21-22
Lines: 96


 Library of Congress to Host Dead Sea Scroll Symposium April 21-22
 To: National and Assignment desks, Daybook Editor
 Contact: John Sullivan, 202-707-9216, or Lucy Suddreth, 202-707-9191
          both of the Library of Congress

   WASHINGTON, April 19  -- A symposium on the Dead Sea 
Scrolls will be held at the Library of Congress on Wednesday,
April 21, and Thursday, April 22.  The two-day program, cosponsored
by the library and Baltimore Hebrew University, with additional
support from the Project Judaica Foundation, will be held in the
library's Mumford Room, sixth floor, Madison Building.
   Seating is limited, and admiss

In [ ]:
pipeline = Pipeline([
    ('countvec', CountVectorizer(binary=True, stop_words='english')),
    ('lr', LogisticRegression(max_iter=1000))
])

# the __ syntax sets parameters for a specific step
param_grid = {
    "countvec__min_df" : [1, 10, 100],
    "lr__C" : [0.01, 1, 10, 100]  # C is the regularization strength of LogisticRegression()
}

grid_search = GridSearchCV(pipeline, param_grid, verbose=1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

print("Predicted labels for the test set using the optimal model:")
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)
print(predictions)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'countvec__min_df': [1, 10, ...], 'lr__C': [0.01, 1, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


Best parameters: {'countvec__min_df': 1, 'lr__C': 10}
Best cross-validation score: 0.9774834437086094
Predicted labels for the test set using the optimal model:
[1 1 1 0 1 1 0 0 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 0 0 0 1 1 1 0 1 1 0 1 0 1 1
 1 0 0 0 1 0 1 0 1 0 0 1 1 0 0 1 1 1 1 1 0 0 1 0 1 0 1 0 1 0 1 0 1 0 1 1 1
 0 1 0 1 1 1 0 1 0 1 0 0 0 1 1 1 0 1 1 0 1 1 1 1 0 0 1 0 0 1 1 0 0 1 1 0 1
 0 1 0 1 0 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 0 1 1 0 1 0 1 1 0
 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 1 0 0 1 0 1 0 0 0 0 1 0 1 1 1 1 0 1 0 1 1 0
 1 0 1 1 1 1 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 1 0 0 1 0 0 0 1 1 1 0 0 0 0 0 0
 0 1 0 1 0 1 1 1 1 1 1 0 1 0 1 0 0 1 0 1 0 0 0 1 1 1 0 1 1 1 1 0 0 0 1 0 1
 0 1 1 0 1 1 0 0 0 1 1 1 1 0 0 0 1 0 0 1 1 1 0 1 0 1 0 1 0 1 1 0 1 0 0 1 1
 0 0 0 0 1 0 0 1 1 1 1 1 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1]


We can extract the model's estimated probability that each test example belongs to each class (sums to 1)

In [56]:
best_model.predict_proba(X_test)

array([[5.93437396e-04, 9.99406563e-01],
       [1.69608906e-04, 9.99830391e-01],
       [2.62136492e-02, 9.73786351e-01],
       [9.96947090e-01, 3.05291031e-03],
       [2.94291155e-03, 9.97057088e-01],
       [2.96276489e-03, 9.97037235e-01],
       [9.99346838e-01, 6.53161819e-04],
       [9.93119252e-01, 6.88074845e-03],
       [1.70079017e-01, 8.29920983e-01],
       [2.49376912e-05, 9.99975062e-01],
       [9.99701000e-01, 2.98999962e-04],
       [4.36110039e-03, 9.95638900e-01],
       [6.85680843e-05, 9.99931432e-01],
       [9.75243532e-03, 9.90247565e-01],
       [1.40549691e-02, 9.85945031e-01],
       [1.95249350e-04, 9.99804751e-01],
       [6.32754722e-03, 9.93672453e-01],
       [9.99794491e-01, 2.05509481e-04],
       [2.25604915e-05, 9.99977440e-01],
       [9.99939956e-01, 6.00444647e-05],
       [9.97901322e-01, 2.09867825e-03],
       [5.20389177e-03, 9.94796108e-01],
       [4.83416621e-03, 9.95165834e-01],
       [9.99771898e-01, 2.28102050e-04],
       [9.997111

We can find the examples which have the highest estimated probability for each class

In [57]:
# most confident in the first class (alt.atheism)
most_confident_first_class = np.argmax(probs[:, 0])
print(X_test[most_confident_first_class])
print(probs[most_confident_first_class, 0])

From: keith@cco.caltech.edu (Keith Allan Schneider)
Subject: Re: <Political Atheists?
Organization: California Institute of Technology, Pasadena
Lines: 191
NNTP-Posting-Host: punisher.caltech.edu

livesey@solntze.wpd.sgi.com (Jon Livesey) writes:

>Much though it might be fun to debate capital punishment itself,
>this is probably the wrong group for it.  The only relevance here
>is that you don't seem to be able to tell us what capital punishment
>actually is, and when it is murder.  That is, when you tell us murder
>is wrong, you are using a term you have not yet defined.

Well, I've said that when an innocent person has been executed, this is
objectively a murder.  However, who is at blame is another question.
It seems that the entire society that sanctions any sorts of executions--
realizing the risks--is to blame.

>There is a *probability* of 
>killing an innocent person by shooting at random into the air, and 
>there is a *probability* of killing an innocent person when the
>stat

In [58]:
# most confident in the second class (soc.religion.christian)
most_confident_second_class = np.argmax(probs[:, 1])
print(X_test[most_confident_second_class])
print(probs[most_confident_second_class, 1])

From: davem@bnr.ca (Dave Mielke)
Subject: Does God love you?
Organization: Bell Northern Research, Ottawa, Canada
Lines: 416

I have come across what I consider to be an excellent tract. It is a
bit lengthy for a posting, but I thought I'd share it with all of you
anyway. Feel free to pass it along to anyone whom you feel might
benefit from what it says. May God richly bless those who read it.
 
 
                   D O E S  G O D  L O V E  Y O U ?
 
 
Q. What  kind  of  question  is that?   Anyone who can read sees signs,
   tracts, books, and bumper stickers that say, "God Loves You."  Isn't
   that true?
 
A. It  is  true that God offers His love to the whole world, as we read
   in one of the most quoted verses in the Bible:
 
      For  God  so  loved  the world, that he gave his only begotten
      Son, that whosoever believeth in him should  not  perish,  but
      have everlasting life.                               John 3:16
 
   However, God's love is qualified.  The Bible sa

We can list the most informative words for predicting the members of each class

In [59]:
vocab = best_model.named_steps['countvec'].get_feature_names_out()
weights = best_model.named_steps['lr'].coef_.ravel()
words_weights_df = pd.DataFrame(data=weights, index=vocab, columns=['Weight'])
words_weights_df.sort_values(by="Weight", ascending=False).head(15)
words_weights_df.sort_values(by="Weight", ascending=True).head(15)

,Weight
rutgers,1.574683
christians,1.110263
christ,1.062733
athos,0.909024
clh,0.901249
1993,0.699385
geneva,0.672112
heaven,0.612227
faith,0.610381
christian,0.598053


,Weight
host,-1.660355
nntp,-1.584364
posting,-1.431034
article,-1.218031
writes,-1.066994
keith,-0.851672
newsreader,-0.829123
atheists,-0.797331
islamic,-0.794510
psuvm,-0.739667


For a given review, we can extract which tokenized words are present

In [60]:
ex = 0        # an arbitrary example  
ex_class = 1  # an arbitrary class  
best_model.predict_proba(X_test)[ex, ex_class]  
X_test[ex]  
vectorizer = best_model.named_steps['countvec']  
vocab = vectorizer.get_feature_names_out()  
words_in_ex = vectorizer.transform([X_test[ex]]).toarray().ravel().astype(bool)  
words_in_ex  
np.sum(words_in_ex)  
np.array(vocab)[words_in_ex]  


np.float64(0.9994065626040718)

'From: noye@midway.uchicago.edu (vera shanti noyes)\nSubject: Re: Am I going to Hell?\nReply-To: noye@midway.uchicago.edu\nOrganization: University of Chicago\nLines: 51\n\nIn article <Apr.24.01.09.10.1993.4254@geneva.rutgers.edu> stoney@oyster.smcm.edu (Stanley Toney) writes:\n> Muslims, i fear have been given a lie from the  \n>fater of lies, Satan. They need Christ as do us all.\n>\n>stan toney stoney@oyster.smcm.edu\n>my opinions are my own, you may borrow them\n\njust picked out this one point because it struck me....\nwhy do you believe this?  muslims believe in many of the same things\nthat christians and jews believe; they believe jesus, while not the\nmessiah, is a prophet.  this seems to me to be much closer to\nchristianity than other religions are.  (then again i tend to be\nsomewhat liberal about others\' beliefs.)\n\nthis also relates to the serbian "ethnic cleansing" question.  i have\nbeen waiting for condemnations of this and have seen very few.  HOW\ncan we stand by a

array([False, False, False, ..., False, False, False], shape=(16279,))

np.int64(154)

array(['01', '09', '10', '1993', '24', '51',
       '_______________________________________________________________________________',
       'accept', 'advocate', 'anti', 'appear', 'appears', 'appropriate',
       'apr', 'article', 'assume', 'beliefs', 'believe', 'bible', 'blood',
       'boil', 'borrow', 'case', 'chest', 'chicago', 'christ',
       'christian', 'christianity', 'christians', 'cleansing', 'clh',
       'closer', 'come', 'comment', 'common', 'condemn', 'converting',
       'did', 'different', 'discuss', 'doesn', 'don', 'drop', 'earlier',
       'edu', 'ethnic', 'fater', 'fear', 'fist', 'geneva', 'given',
       'going', 'got', 'group', 'hand', 'hell', 'honor', 'hopes', 'ill',
       'innocent', 'islam', 'isn', 'issue', 'jesus', 'jews', 'just',
       'killed', 'killing', 'kindness', 'know', 'koran', 'liberal', 'lie',
       'lies', 'line', 'lines', 'love', 'loves', 'make', 'maybe', 'means',
       'messiah', 'midway', 'mood', 'moslem', 'muslims', 'need', 'noye',
       

We can explicitly compute a linear combination of the learned weights and input features to assess how confident the classifier is (i.e., the larger the absolute value of this number is, the more confidence the model has that it belongs to the predicted class)

In [61]:
ex = 0  # defines the example
words_in_ex = vectorizer.transform([X_test[ex]]).toarray().ravel().astype(bool)
weights = best_model.named_steps['lr'].coef_[0]
ex_df = pd.DataFrame(data=weights[words_in_ex], index=np.array(vocab)[words_in_ex], columns=['Weight'])
ex_df.sum()
ex_df

Weight    6.768337
dtype: float64

,Weight
01,-0.077807
09,0.201211
10,0.342832
1993,0.699385
24,0.018300
...,...
ways,0.057383
women,-0.120666
worried,-0.008256
writes,-1.066994


#### Example 3

We'll illustrate word embeddings using Google's `word2vec` model. For this example to work properly, you'll need to manually download the model from https://code.google.com/archive/p/word2vec/ and install the `gensim` and `pot` packages in your Python environment

In [ ]:
# allow multiple outputs in one cell
if True:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = "all"

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [20]:
import pandas as pd
import gensim
from gensim.models import KeyedVectors

In [7]:
model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)
print('Size of vocabulary: ', len(model.index_to_key))

Size of vocabulary:  3000000


We can find the similarity between two words

In [9]:
word_pairs = [('pineapple', 'mango'), ('sun', 'robot')]

for pair in word_pairs:
    similarity = model.similarity(pair[0], pair[1])
    print(f"The similarity between '{pair[0]}' and '{pair[1]}' is {similarity:.3f}")

The similarity between 'pineapple' and 'mango' is 0.668
The similarity between 'sun' and 'robot' is 0.029


We can find the vocabulary words which are most similar to a target word

In [16]:
model.most_similar('UBC')  # University of British Columbia

[('UVic', 0.788647472858429),
 ('SFU', 0.7588528394699097),
 ('Simon_Fraser', 0.7356574535369873),
 ('UFV', 0.6880435943603516),
 ('VIU', 0.6778583526611328),
 ('Kwantlen', 0.677142858505249),
 ('UBCO', 0.6734487414360046),
 ('UPEI', 0.6731126308441162),
 ('UBC_Okanagan', 0.6709135174751282),
 ('Lakehead_University', 0.6622507572174072)]

We can find the misplaced word in a sequence

In [12]:
print(model.doesnt_match("sun moon earth UBC mars".split()))

UBC


We can measure the "distance" between sentences

In [17]:
sentence_obama = 'Obama speaks to the media in Illinois'.lower().split()
sentence_president = 'The president greets the press in Chicago'.lower().split()
sentence_unrelated = 'Data science is a multidisciplinary blend of data inference, algorithmm development, and technology.'

similarity = model.wmdistance(sentence_obama, sentence_president)
print("Distance between related sentences {:.4f}".format(similarity))

similarity = model.wmdistance(sentence_obama, sentence_unrelated)
print("Distance between unrelated sentences {:.4f}".format(similarity))

Distance between related sentences 0.7331
Distance between unrelated sentences 1.2634


In [24]:
def analogy(word1, word2, word3):
    print('"%s" is to "%s" as "%s" is to ?' %(word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=['Analogy word', 'Score'])

display(analogy('man','king','woman'))
display(analogy('Montreal', 'Canadiens', 'Vancouver'))
display(analogy('man', 'computer_programmer', 'woman'))  # note the bias

"man" is to "king" as "woman" is to ?


,Analogy word,Score
0,queen,0.711819
1,monarch,0.618967
2,princess,0.590243
3,crown_prince,0.549946
4,prince,0.537732
5,kings,0.523684
6,Queen_Consort,0.523595
7,queens,0.518113
8,sultan,0.509859
9,monarchy,0.508741


"Montreal" is to "Canadiens" as "Vancouver" is to ?


,Analogy word,Score
0,Canucks,0.821327
1,Vancouver_Canucks,0.750401
2,Calgary_Flames,0.705470
3,Leafs,0.695783
4,Maple_Leafs,0.691617
5,Thrashers,0.687504
6,Avs,0.681716
7,Sabres,0.665307
8,Blackhawks,0.664625
9,Habs,0.661023


"man" is to "computer_programmer" as "woman" is to ?


,Analogy word,Score
0,homemaker,0.562712
1,housewife,0.510505
2,graphic_designer,0.505180
3,schoolteacher,0.497949
4,businesswoman,0.493489
5,paralegal,0.492551
6,registered_nurse,0.490797
7,saleswoman,0.488163
8,electrical_engineer,0.479773
9,mechanical_engineer,0.475540


The general idea is that each word has a position in a vector space where the distances between words tell you something about their conceptual relationship. For example, you can do something like $\text{QUEEN}=\text{KING}-\text{MAN}+\text{WOMEN}$ as a vector operation in that space

You can perform a similar analysis with other packages such as `spacy`